# 台指期資料爬蟲

In [ ]:
%pip install selenium

In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from datetime import datetime, timedelta
import os
import csv

# setting #######################################################
end_year = 2024                         # 資料抓到這一年
csv_file_path = path = os.path.join(os.path.join(os.path.abspath(os.path.dirname(os.getcwd()))), 'history_data','tw','twii_history.csv')     # 有過去資料的檔案
#################################################################

# 用with語句開啟檔案，確保在使用完後自動關閉
with open(csv_file_path, 'r') as file:
    # 使用csv.reader讀取檔案
    reader = csv.reader(file)

    # 將讀取的內容轉換為列表，並獲取最後一行
    last_row = list(reader)[-1]

########################################
# 直接設定開始日期
start_date = last_row[0]
########################################

y = str(start_date)[:4]
m = int(str(start_date)[4:6])
d = str(start_date)[6:]

url = 'https://www.twse.com.tw/en/indices/taiex/mi-5min-hist.html'
driver = webdriver.Chrome()
driver.get(url)
driver.find_element(By.XPATH, '//*[@id="popupStatement"]/div/div/button').click()

for k in range(int(y), end_year+1):
    for j in range(m, 13):

        input_search_y = Select(driver.find_element(By.XPATH, '//*[@id="label0"]'))
        input_search_y.select_by_value(str(k))

        input_search_y = Select(driver.find_element(By.XPATH, '//*[@id="form"]/div/div[1]/span/select[2]'))
        input_search_y.select_by_value(str(j))

        search_btn = driver.find_element(By.XPATH, '//*[@id="form"]/div/button')
        search_btn.click()

        driver.implicitly_wait(1)
        sheet = driver.find_elements(By.XPATH, '//*[@id="reports"]/div[2]/div[2]/table/tbody/tr')
        print(len(sheet))

        for i in range(len(sheet)):
            # 設定時間避免重疊
            d = driver.find_element(By.XPATH, '//*[@id="reports"]/div[2]/div[2]/table/tbody/tr[' + str(i+1) + ']/td[1]').text
            original_date = datetime.strptime(d, '%Y/%m/%d')
            new_date_str = original_date.strftime('%Y%m%d')
            if int(start_date) >= int(new_date_str):
                continue

            data = [0, 0, 0, 0, 0]
            data[0] = new_date_str
            data[1] = driver.find_element(By.XPATH, '//*[@id="reports"]/div[2]/div[2]/table/tbody/tr[' + str(i+1) + ']/td[2]').text.replace(',', '')
            data[2] = driver.find_element(By.XPATH, '//*[@id="reports"]/div[2]/div[2]/table/tbody/tr[' + str(i+1) + ']/td[3]').text.replace(',', '')
            data[3] = driver.find_element(By.XPATH, '//*[@id="reports"]/div[2]/div[2]/table/tbody/tr[' + str(i+1) + ']/td[4]').text.replace(',', '')
            data[4] = driver.find_element(By.XPATH, '//*[@id="reports"]/div[2]/div[2]/table/tbody/tr[' + str(i+1) + ']/td[5]').text.replace(',', '')

            print(data)

            with open(csv_file_path, 'a', newline='') as csvfile:
                # 建立 CSV 檔寫入器
                writer = csv.writer(csvfile)

                # 寫入一列資料
                writer.writerow(data)

            # time.sleep(2)
        m = 1

driver.quit()

22
['20241227', '23251.28', '23335.66', '23204.35', '23275.68']
['20241230', '23274.26', '23331.67', '23159.70', '23190.20']
['20241231', '23094.07', '23123.96', '22972.99', '23035.10']


# 計算技術指標的值

In [9]:
import pandas as pd
import numpy as np
import os

path = os.path.join(os.path.abspath(os.path.dirname(os.getcwd())), 'history_data', 'tw')
print(path)


df = pd.read_csv(path + "/twii_history.csv", encoding='utf-8')

# 計算移動均線 (SMA)
df['MA5'] = df['收盤指數'].rolling(window=5).mean()
df['MA10'] = df['收盤指數'].rolling(window=10).mean()
df['MA20'] = df['收盤指數'].rolling(window=20).mean()

# 計算指數移動平均線 (EMA)
df['EMA5'] = df['收盤指數'].ewm(span=5, adjust=False).mean()
df['EMA12'] = df['收盤指數'].ewm(span=12, adjust=False).mean()
df['EMA26'] = df['收盤指數'].ewm(span=26, adjust=False).mean()

# 計算MACD和DIF
df['DIF'] = df['EMA12'] - df['EMA26']
df['MACD'] = df['DIF'].ewm(span=9, adjust=False).mean()


# 計算RSI
def calculate_rsi(series, period=14):
    delta = series.diff(1)
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

df['RSI'] = calculate_rsi(df['收盤指數'])

# 計算KDJ
low_list = df['最低指數'].rolling(9, min_periods=9).min()
low_list.fillna(value=df['最低指數'].expanding().min(), inplace=True)
high_list = df['最高指數'].rolling(9, min_periods=9).max()
high_list.fillna(value=df['最高指數'].expanding().max(), inplace=True)
rsv = (df['收盤指數'] - low_list) / (high_list - low_list) * 100


# 初始值設定
df['K'] = 50
df['D'] = 50

for i in range(1, len(df)):
    df.loc[df.index[i], 'K'] = (2/3) * df.loc[df.index[i-1], 'K'] + (1/3) * rsv[i]
    df.loc[df.index[i], 'D'] = (2/3) * df.loc[df.index[i-1], 'D'] + (1/3) * df.loc[df.index[i], 'K']

# Rename columns
df.rename(columns={'日期': 'Date', '開盤指數': 'Open', '最高指數': 'High', '最低指數': 'Low', '收盤指數': 'Close'}, inplace=True)


# 顯示DataFrame
df.to_csv(path + "/stock_price/twiitech.csv", encoding='utf-8', index=False)


/Users/yanyifu/Documents/_Coding/LLM Predict Stock/history_data/tw


/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_18997/1754226814.py:50: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '55.240163383485836' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[df.index[i], 'K'] = (2/3) * df.loc[df.index[i-1], 'K'] + (1/3) * rsv[i]
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_18997/1754226814.py:51: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '51.74672112782861' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[df.index[i], 'D'] = (2/3) * df.loc[df.index[i-1], 'D'] + (1/3) * df.loc[df.index[i], 'K']
